# Autoencoder disperso, Mirai

Autoencoder denso con una penalización L1 sobre las activaciones del cuello de botella, que favorece una representación dispersa. La dirección IP de origen (`src_ip`) no se usa como entrada, porque identifica a los equipos atacantes.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path("../../data/flows.csv")

# class_name revela el tipo de ataque y flow_start es una marca de tiempo absoluta
df = pd.read_csv(DATA_PATH).drop(columns=['class_name', 'flow_start'])

print(df.shape)
df.head()

(1006776, 11)


,src_ip,duration,state,n_pkts,n_bytes,f_pkts,f_bytes,b_pkts,b_bytes,avg_pkt_size,label
0,b'\xc0\xa8\xa3\x8a',296.702183,TCP_OTHER,20,1200,20,1200,0,0,60.000000,1
1,b'\xc0\xa8\xa3\x8a',32.005860,TCP_OTHER,40,2775,20,1429,20,1346,69.375000,1
2,b'\xc0\xa8\xa3\x8a',19.642577,TCP_OTHER,24,1678,12,868,12,810,69.916667,1
3,b'\xc0\xa8\xa3\x9b',19.606732,TCP_OTHER,22,1546,11,802,11,744,70.272727,1
4,b'\xc0\xa8\xa3\x8b',289.523967,TCP_OTHER,39,2340,26,1560,13,780,60.000000,1


In [3]:
y = df['label'].values
X = pd.get_dummies(df.drop(columns=['label', 'src_ip']), columns=['state'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (805420, 15), X_train_normal: (6234, 15), X_test: (201356, 15)


In [4]:
input_dim = X_train.shape[1]

inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(32, activation='relu')(x)
bottleneck = layers.Dense(16, activation='relu', activity_regularizer=regularizers.l1(1e-5))(x)
x = layers.Dense(32, activation='relu')(bottleneck)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(input_dim, activation='sigmoid')(x)

autoencoder = keras.Model(inputs, outputs)
encoder = keras.Model(inputs, bottleneck)
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
    verbose=0,
)
print(f"Epocas entrenadas: {len(history.history['loss'])}, val_loss final: {history.history['val_loss'][-1]:.5f}")

Epocas entrenadas: 100, val_loss final: 0.00030


In [5]:
def reconstruction_error(X):
    return np.mean(np.square(X - autoencoder.predict(X, verbose=0)), axis=1)

threshold = np.percentile(reconstruction_error(X_train_normal), 95)
y_pred = (reconstruction_error(X_test) > threshold).astype(int)

print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, reconstruction_error(X_test)):.4f}")

bottleneck_act = encoder.predict(X_test, verbose=0)
sparsity = np.mean(bottleneck_act < 1e-3)
print(f"Sparsity del bottleneck (activaciones ~0): {sparsity*100:.1f}%")

              precision    recall  f1-score   support

      normal       0.04      0.96      0.08      1559
      ataque       1.00      0.84      0.91    199797

    accuracy                           0.84    201356
   macro avg       0.52      0.90      0.50    201356
weighted avg       0.99      0.84      0.91    201356

[[  1489     70]
 [ 32152 167645]]


ROC-AUC: 0.9358


Sparsity del bottleneck (activaciones ~0): 54.2%
